In [0]:
%run ./00_config

In [0]:
from pyspark.sql import functions as F

def ingest_bronze(feed, fmt, options=None):
    table = f"bronze_{feed}"
    before = spark.table(table).count() if spark.catalog.tableExists(table) else 0
    reader = (spark.readStream
                .format("cloudFiles")
                .option("cloudFiles.format", fmt)
                .option("cloudFiles.schemaLocation", f"{CHK}/bronze_{feed}/_schema"))
    for key, value in (options or {}).items():
        reader = reader.option(key, value)

    df = (reader.load(f"{RAW}/{feed}/")
            .withColumn("_source_file",      F.col("_metadata.file_path"))
            .withColumn("_file_modified_at", F.col("_metadata.file_modification_time"))
            .withColumn("_ingested_at",      F.current_timestamp()))

    query = (df.writeStream
               .option("checkpointLocation", f"{CHK}/bronze_{feed}")
               .trigger(availableNow=True)
               .toTable(f"bronze_{feed}"))
    query.awaitTermination()

    after = spark.table(f"bronze_{feed}").count()
    print(f"bronze_{feed}: {after - before} new rows ({after} total)")

In [0]:
for feed in ["assets", "clients", "prices", "trades"]:
    ingest_bronze(feed, "csv", {"header": "true"})

ingest_bronze("ticks", "json")